In [1]:
import pandas as pd

In [5]:
df = pd.read_csv(r"C:\Users\DELL\Desktop\E-Commerce-Project\data.csv", encoding='ISO-8859-1')
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12-01-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12-01-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12-01-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12-01-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12-01-2010 08:26,3.39,17850.0,United Kingdom


In [10]:
# --- Initial data check ---
print(df.shape)      # Confirms row/column count matches raw SQL table (541909, 8)
df.info()            # Shows column types and non-null counts - spot missing values early

(541909, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [16]:
# --- Clean the data: remove rows with missing CustomerID ---
# .copy() ensures df_clean is a fully independent dataframe

df_clean = df[df.CustomerID.notna()].copy()

In [17]:
df_clean.shape

(406829, 8)

In [83]:
# --- Convert InvoiceDate from text to proper datetime ---

df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], format='mixed')

In [19]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    406829 non-null  object        
 1   StockCode    406829 non-null  object        
 2   Description  406829 non-null  object        
 3   Quantity     406829 non-null  int64         
 4   InvoiceDate  406829 non-null  datetime64[ns]
 5   UnitPrice    406829 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      406829 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 27.9+ MB


In [20]:
# --- Verify no stray "adjustment rows" remain (negative qty, no "C" invoice, already had null CustomerID) ---

len(df_clean[(df_clean['Quantity'] < 0) & (~df_clean['InvoiceNo'].str.startswith('C'))])

0

In [27]:
# --- Add IsCancellation flag ---
# True/False -> converted to 1/0 integers to match SQL's IsCancellation column

df_clean['IsCancellation'] = df_clean['InvoiceNo'].str.startswith('C').astype(int)

In [84]:
df_clean['IsCancellation'].head(5)

In [31]:
# Verify cancellation row count matches SQL (8905)

len(df_clean[(df_clean['IsCancellation'] == 1)])

8905

In [33]:
# total spend ---

df_clean['Total_Spend'] = df_clean['Quantity'] * df_clean['UnitPrice']

In [34]:
df_clean[['Quantity', 'UnitPrice', 'Total_Spend']].head()

,Quantity,UnitPrice,Total_Spend
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


In [38]:
# --- Calculate Monetary: total spend per customer ---

monetary = df_clean.groupby('CustomerID')['Total_Spend'].sum()

In [41]:
# --- Calculate Frequency: distinct orders per customer ---
# .nunique() counts DISTINCT invoice numbers, not every row (avoids counting multi-item orders multiple times)

frequency = df_clean.groupby('CustomerID')['InvoiceNo'].nunique()
len(frequency)

4372

In [48]:
# --- Calculate Recency: days since each customer's last purchase ---

overall_recent_date = df_clean['InvoiceDate'].max()
overall_recent_date

Timestamp('2011-12-09 12:50:00')

In [49]:
last_purchase = df_clean.groupby('CustomerID')['InvoiceDate'].max()
last_purchase

CustomerID
12346.0   2011-01-18 10:17:00
12347.0   2011-12-07 15:52:00
12348.0   2011-09-25 13:13:00
12349.0   2011-11-21 09:51:00
12350.0   2011-02-02 16:01:00
                  ...        
18280.0   2011-03-07 09:52:00
18281.0   2011-06-12 10:53:00
18282.0   2011-12-02 11:43:00
18283.0   2011-12-06 12:02:00
18287.0   2011-10-28 09:29:00
Name: InvoiceDate, Length: 4372, dtype: datetime64[ns]

In [51]:
recency = (overall_recent_date - last_purchase).dt.days
recency

CustomerID
12346.0    325
12347.0      1
12348.0     74
12349.0     18
12350.0    309
          ... 
18280.0    277
18281.0    180
18282.0      7
18283.0      3
18287.0     42
Name: InvoiceDate, Length: 4372, dtype: int64

In [55]:
# --- Combine R, F, M into a single dataframe ---

rfm = pd.concat([recency, frequency, monetary], axis=1)
rfm = rfm.rename(columns={'InvoiceDate': 'recency', 'InvoiceNo': 'frequency', 'Total_Spend': 'monetary'})
rfm.head()

,recency,frequency,monetary
CustomerID,,,
12346.0,325,2,0.00
12347.0,1,7,4310.00
12348.0,74,4,1797.24
12349.0,18,1,1757.55
12350.0,309,1,334.40


pd.qcut() - It looks at all the values in column, sorts them, and splits them into 5 equal-sized groups (quintiles) — the 20% smallest values get one label, the next 20% get the next label, and so on.

In [69]:
# --- Frequency Score (F_Score) ---

rfm_sorted = rfm.sort_values(by=['frequency', 'CustomerID'])
rfm_sorted['F_rank'] = rfm_sorted['frequency'].rank(method='first')
rfm_sorted['F_Score'] = pd.qcut(rfm_sorted['F_rank'], q=5, labels=[1,2,3,4,5])

In [85]:
# --- Monetary Score (M_Score) ---

rfm_sorted = rfm_sorted.sort_values(by=['monetary', 'CustomerID'])
rfm_sorted['M_rank'] = rfm_sorted['monetary'].rank(method='first')
rfm_sorted['M_Score'] = pd.qcut(rfm_sorted['M_rank'], q=5, labels=[1,2,3,4,5])

In [71]:
rfm_sorted[['recency', 'frequency', 'monetary', 'F_Score', 'M_Score']].head(10)

,recency,frequency,monetary,F_Score,M_Score
CustomerID,,,,,
17448.0,144,1,-4287.63,2,1
15369.0,143,1,-1592.49,1,1
14213.0,371,1,-1192.20,1,1
17603.0,49,5,-1165.30,4,1
12503.0,337,1,-1126.00,1,1
15823.0,336,2,-840.76,3,1
13154.0,143,1,-611.86,1,1
15802.0,142,3,-451.42,3,1
16252.0,365,1,-295.09,1,1


In [74]:
# --- Recency Score (R_Score) ---
# Reversed labels: low recency (recent buyer) = best = 5, high recency = worst = 1

rfm_sorted = rfm_sorted.sort_values(by=['recency', 'CustomerID'])
rfm_sorted['R_rank'] = rfm_sorted['recency'].rank(method='first')
rfm_sorted['R_Score'] = pd.qcut(rfm_sorted['R_rank'], q=5, labels=[5,4,3,2,1])

In [75]:
rfm_sorted[['recency', 'frequency', 'monetary', 'F_Score', 'M_Score', 'R_Score']].head(10)

,recency,frequency,monetary,F_Score,M_Score,R_Score
CustomerID,,,,,,
12423.0,0,9,1849.11,5,4,5
12433.0,0,7,13375.87,4,5,5
12476.0,0,20,6546.58,5,5,5
12518.0,0,5,2056.89,4,5,5
12526.0,0,3,1316.66,3,4,5
12662.0,0,12,3817.08,5,5,5
12680.0,0,4,862.81,4,3,5
12713.0,0,1,848.55,1,3,5
12748.0,0,224,29072.10,5,5,5


In [76]:
# --- RFM_Code (concatenate scores as text)

rfm_sorted['RFM_Code'] = (
        rfm_sorted['R_Score'].astype(str) +
        rfm_sorted['F_Score'].astype(str) +
        rfm_sorted['M_Score'].astype(str)
)



In [77]:
# --- Cust_Status

def get_status(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Loyal'
    elif r == 5 and f == 1 and m == 5:
        return 'Big Spender'
    elif r == 5 and f == 1:
        return 'New Customer'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Dormant'
    else:
        return 'At_Risk'

In [80]:
# Scores are currently 'category' dtype from qcut — convert to int first

rfm_sorted['R_Score'] = rfm_sorted['R_Score'].astype(int)
rfm_sorted['F_Score'] = rfm_sorted['F_Score'].astype(int)
rfm_sorted['M_Score'] = rfm_sorted['M_Score'].astype(int)

rfm_sorted['Cust_Status'] = rfm_sorted.apply(get_status, axis=1)

In [81]:
# --- Decile (NTILE(10) on monetary, same rank-based approach) ---

rfm_sorted = rfm_sorted.sort_values(by=['monetary', 'CustomerID'])
rfm_sorted['Decile_rank'] = rfm_sorted['monetary'].rank(method='first', ascending=False)
rfm_sorted['Decile'] = pd.qcut(rfm_sorted['Decile_rank'], q=10, labels=list(range(1,11)))

In [82]:
# --- Final check: compare segment counts to your Power BI donut chart ---

print(rfm_sorted['Cust_Status'].value_counts())

Cust_Status
At_Risk         2556
Loyal            964
Dormant          811
New Customer      40
Big Spender        1
Name: count, dtype: int64
